# 📥 Laboratório de ingestão e entrega (`ingest/`) — batch vs NRT

**O que este notebook é:** a mecânica da ingestão desmontada — o evento limpo, os defeitos com probabilidade conhecida, as duplicatas "at least once", a serialização JSONL+gzip da bronze e, na parte NRT, a **semântica** de particionamento do Kinesis (por que a PartitionKey é uma decisão, e como nasce um hot shard) — tudo simulado em memória.

**O que ele não é:** o caminho NRT de verdade (Kinesis → Firehose → bronze). Esse roda com `make nrt` com o LocalStack de pé; aqui exploramos a *lógica*, que é o que se discute em entrevista.

**Curiosidade útil:** este notebook não usa Spark — ingestão neste repo é Python puro (`boto3` + stdlib), e a bancada respeita isso.

**Par no projeto:** Passos 3 (geração e ingestão) e 9 (NRT); o hot shard conversa direto com o skew do Passo 8.

**Sumário**
1. Setup
2. Anatomia do evento limpo
3. `corrupt`: defeitos com probabilidade conhecida
4. `build_day`: duplicatas "at least once"
5. A serialização da bronze: JSONL + gzip em memória
6. NRT: PartitionKey, ordem por chave e o hot shard
7. Exercícios

> Convenção da pasta: roda de cima a baixo; seed fixa; sem infraestrutura de pé.

## 1. Setup

Só stdlib + os módulos do repo. O `boto3` precisa estar instalado porque o módulo do gerador o importa no topo — mas nenhuma célula aqui toca S3.

In [ ]:
import sys

sys.path.insert(0, "../ingest")

import gzip
import io
import json
import random
from collections import Counter

from generate_events import build_day, clean_event, corrupt

rng = random.Random(42)
merchant_ids = [f"mer_{i:05d}" for i in range(1, 301)]
DT = "2026-07-03"
print("setup ok —", len(merchant_ids), "merchants")

## 2. Anatomia do evento limpo

`clean_event` é a "linha perfeita" do gerador: todos os campos válidos, `occurred_at` sorteado dentro do dia, id determinístico a partir da seed. É **sobre** esta base que os defeitos serão injetados — a mesma filosofia do `_row()` dos testes.

In [ ]:
evento = clean_event(rng, DT, merchant_ids)
print(json.dumps(evento, indent=2, ensure_ascii=False))

## 3. `corrupt`: defeitos com probabilidade conhecida

O `corrupt` injeta **no máximo um** defeito por evento, numa cadeia de probabilidades fixas (veja o código-fonte): ~2,0% amount inválido, ~1,2% moeda fora do domínio, ~1,0% timestamp impossível, ~0,8% customer nulo. Cada defeito mapeia para **uma regra do contrato** do Passo 4 — o gerador e o contrato são as duas pontas do mesmo fio.

Vamos medir: 20.000 eventos, comparando cada um com sua cópia antes do `corrupt`:

In [ ]:
N = 20_000
contagem = Counter()
for _ in range(N):
    original = clean_event(rng, DT, merchant_ids)
    mutado = corrupt(dict(original), rng)
    if mutado["amount_cents"] != original["amount_cents"]:
        contagem["amount_invalido"] += 1
    elif mutado["currency"] != original["currency"]:
        contagem["moeda_fora_dominio"] += 1
    elif mutado["occurred_at"] != original["occurred_at"]:
        contagem["timestamp_invalido"] += 1
    elif mutado["customer_id"] != original["customer_id"]:
        contagem["customer_id_nulo"] += 1
    else:
        contagem["limpo"] += 1

for defeito, n in contagem.most_common():
    print(f"{defeito:20s} {n:6d}  ({n / N:.2%})")

As frequências observadas ecoam as probabilidades do código — e a soma dos defeitos (~5%) explica por que o limite do quality gate (`MAX_REJECT_RATE=0.05`) é o que é: o gerador foi calibrado para viver **no limite** do gate. Rode a seção 5 do notebook 01 e compare o `reject_rate` de lá com a soma daqui.

## 4. `build_day`: duplicatas "at least once"

Broker de mensagens honesto promete entrega *at least once* — ou seja, duplicata **vai** acontecer. O `build_day` simula isso re-emitindo ~3% dos eventos como cópias exatas e embaralhando tudo:

In [ ]:
dia = build_day(rng, DT, merchant_ids, volume=10_000)
ids_unicos = len({e["event_id"] for e in dia})
print(f"eventos no dia: {len(dia)} | event_ids unicos: {ids_unicos} | "
      f"duplicatas: {len(dia) - ids_unicos} ({(len(dia) - ids_unicos) / ids_unicos:.2%})")

É essa sujeira — deliberada e medida — que a dedup determinística do contrato limpa no Passo 4. Duplicata exata é o caso *fácil*; a versão difícil (mesmo id, conteúdo novo) é o CDC, que tem notebook próprio (`04_laboratorio_cdc`).

## 5. A serialização da bronze: JSONL + gzip em memória

A bronze guarda o **formato de chegada**: JSON Lines comprimido, sem conversão colunar (Parquet é papel do silver). A mesma serialização do `upload_day`, sem o `PutObject`:

In [ ]:
cru = "".join(json.dumps(e, ensure_ascii=False) + "\n" for e in dia).encode("utf-8")

buffer = io.BytesIO()
with gzip.GzipFile(fileobj=buffer, mode="wb") as gz:
    gz.write(cru)
comprimido = buffer.getvalue()

print(f"JSONL cru: {len(cru) / 1024:.0f} KiB | gzip: {len(comprimido) / 1024:.0f} KiB "
      f"| taxa: {len(cru) / len(comprimido):.1f}x")

JSON repete as chaves em toda linha — por isso o gzip rende tanto. É também por isso que a bronze textual é cara de *consultar* (o Spark lê tudo pra achar qualquer coisa) e o silver vira Parquet: colunar, tipado e com predicado empurrado pro arquivo.

## 6. NRT: PartitionKey, ordem por chave e o hot shard

No caminho NRT (`ingest/nrt_producer.py`), cada evento vai pro Kinesis com `PartitionKey = merchant_id`. A chave decide **o shard** — e o Kinesis garante ordem **só dentro do shard**. Escolher a chave de negócio preserva a ordem por estabelecimento (o que a dedup temporal agradece), mas cria um risco: tráfego concentrado num merchant vira tráfego concentrado num shard.

Simulação com 4 shards — primeiro com tráfego uniforme entre os 300 merchants, depois com um merchant dominante levando 40% do volume:

In [ ]:
import hashlib

SHARDS = 4


def shard_de(chave):
    # hash estavel de proposito: o hash() nativo de strings muda a cada
    # processo Python (salt de seguranca) e tornaria a simulacao irreproduzivel
    return int(hashlib.md5(chave.encode()).hexdigest(), 16) % SHARDS


def distribui(eventos):
    porcada = Counter(shard_de(e["merchant_id"]) for e in eventos)
    total = sum(porcada.values())
    for shard in range(SHARDS):
        n = porcada.get(shard, 0)
        barra = "#" * round(40 * n / total)
        print(f"shard {shard}: {n:6d}  {barra}")


print("trafego uniforme (300 merchants):")
distribui(dia)

In [ ]:
dominante = merchant_ids[0]
skew = [dict(e, merchant_id=dominante) if rng.random() < 0.40 else e for e in dia]

print(f"trafego com '{dominante}' levando ~40% do volume:")
distribui(skew)

O shard do merchant dominante vira o gargalo: throttling nas escritas, consumidor atrasado, latência do "até 60s" do Firehose esticando — enquanto os outros shards ficam ociosos. É o **skew do Passo 8, na versão streaming**: mesma doença (chave concentrada), outro órgão (shard em vez de partição Spark).

Os remédios também rimam: *salting* da chave (`merchant_id#0..3`) espalha o tráfego — ao custo de **perder a ordem por merchant**, que era justamente o motivo da chave. Não existe resposta grátis; existe trade-off documentado (é por isso que a escolha está comentada no próprio `nrt_producer.py`).

## 7. Exercícios

1. **Calibre o gate:** dobre mentalmente a probabilidade de `amount_invalido` no `corrupt` (4%). O lote do dia ainda passa no quality gate de 5%? Confirme medindo como na seção 3.
2. **Salting na prática:** reimplemente `shard_de` com `f"{merchant_id}#{rng.randint(0, 3)}"` e redistribua o tráfego skewed. O que aconteceu com o hot shard? E o que você perdeu (pense na dedup temporal)?
3. **Dimensionamento:** um shard do Kinesis aceita 1 MiB/s de escrita. Com o tamanho médio de evento da seção 5, quantos eventos/s cabem num shard — e quantos shards o pico de 10x isso exigiria?
4. **Feche o ciclo:** a duplicata do `build_day` está protegida por teste (`test_deduplicacao_mantem_mais_recente`). O comportamento do `corrupt` (no máximo UM defeito por evento) está? Se não, escreva-o.

*Fim — este notebook não abre SparkSession, então não há o que desligar. O caminho NRT real (Kinesis → Firehose → bronze em até 60s) roda com `make nrt` com o LocalStack de pé; confira a entrega com `make visao`.*